# Day 11 – Temporal Preprocessing Pipeline

## Objective

The objective of Day 11 is to implement a reusable preprocessing pipeline for the timestamped water-quality dataset.

The pipeline includes:

- Timestamp validation and preprocessing
- Duplicate timestamp handling
- Chronological sorting
- Missing-value handling
- Data-quality filtering
- Chronological train/validation/test splitting
- Feature scaling
- Preservation of raw and processed datasets

The original raw dataset will remain unchanged.

Because the original dataset does not contain real timestamps, the timestamps used here were synthetically generated according to the methodology documented during Day 8. Therefore, the temporal preprocessing and forecasting workflow is treated as a methodological experiment rather than a representation of actual sensor deployment.

In [2]:
import pandas as pd
import numpy as np

processed_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\water_quality_timestamped.csv"

df = pd.read_csv(processed_path)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df = df.sort_values("timestamp").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (3276, 11)
Columns:
['timestamp', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,2025-01-01 00:00:00,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,2025-01-01 01:00:00,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,2025-01-01 02:00:00,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,2025-01-01 03:00:00,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,2025-01-01 04:00:00,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


## 1. Timestamp Preprocessing

The timestamp column is converted into a datetime format and records are sorted chronologically.

Invalid timestamps are identified and removed because temporal ordering is required for downstream time-series processing.

In [3]:
print("Invalid timestamps:", df["timestamp"].isna().sum())

df = df.dropna(subset=["timestamp"]).copy()

df = df.sort_values("timestamp").reset_index(drop=True)

print("Shape after timestamp validation:", df.shape)
print("Chronologically sorted:", df["timestamp"].is_monotonic_increasing)

Invalid timestamps: 0
Shape after timestamp validation: (3276, 11)
Chronologically sorted: True


## 2. Duplicate Timestamp Handling

Duplicate timestamps are checked before modeling.

For this initial preprocessing version, duplicate timestamps are retained for audit purposes and then reduced to one observation per timestamp using the first available record.

This provides a deterministic preprocessing rule while keeping the original raw and timestamped datasets unchanged.

In [4]:
duplicate_count = df["timestamp"].duplicated().sum()

print("Duplicate timestamps before cleaning:", duplicate_count)

df = (
    df.sort_values("timestamp")
      .drop_duplicates(subset=["timestamp"], keep="first")
      .reset_index(drop=True)
)

print("Duplicate timestamps after cleaning:",
      df["timestamp"].duplicated().sum())

print("Shape after duplicate handling:", df.shape)

Duplicate timestamps before cleaning: 2
Duplicate timestamps after cleaning: 0
Shape after duplicate handling: (3274, 11)


## 3. Missing-Value Handling

Missing values are identified for each parameter.

For this first preprocessing version, numeric missing values are filled using forward filling because the data is treated as an ordered temporal dataset.

If a missing value occurs at the beginning of the dataset and cannot be forward-filled, the remaining missing value is filled using the training-data median during the modeling pipeline.

No information from future observations is used for temporal feature construction.

In [5]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Missing values before handling:")
display(df[numeric_columns].isna().sum())

Missing values before handling:


ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

In [6]:
df[numeric_columns] = df[numeric_columns].ffill()

print("Missing values after forward filling:")
display(df[numeric_columns].isna().sum())

Missing values after forward filling:


ph                 1
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64

## 4. Basic Data-Quality Filtering

Physically impossible negative values are treated as invalid for parameters that cannot logically be negative.

The target column `Potability` is expected to contain binary values 0 and 1.

Invalid observations are removed before model training.

In [7]:
non_negative_columns = [
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

for column in non_negative_columns:
    if column in df.columns:
        df = df[df[column] >= 0]

if "ph" in df.columns:
    df = df[
        (df["ph"] >= 0) &
        (df["ph"] <= 14)
    ]

if "Potability" in df.columns:
    df = df[df["Potability"].isin([0, 1])]

df = df.reset_index(drop=True)

print("Shape after basic quality filtering:", df.shape)

Shape after basic quality filtering: (3273, 11)


## 5. Chronological Train / Validation / Test Split

Random splitting is avoided because it can introduce future information into the training data.

The dataset is divided chronologically:

- 70% training
- 15% validation
- 15% testing

The earliest observations are used for training, followed by validation observations and finally the test observations.

This split is intended for the initial temporal modeling workflow and should be confirmed with the mentor before being treated as the final project split.

In [8]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain period:")
print(train_df["timestamp"].min(), "→", train_df["timestamp"].max())

print("\nValidation period:")
print(val_df["timestamp"].min(), "→", val_df["timestamp"].max())

print("\nTest period:")
print(test_df["timestamp"].min(), "→", test_df["timestamp"].max())

Train shape: (2291, 11)
Validation shape: (491, 11)
Test shape: (491, 11)

Train period:
2025-01-01 01:00:00 → 2025-04-07 00:45:00

Validation period:
2025-04-07 01:45:00 → 2025-04-27 14:45:00

Test period:
2025-04-27 15:45:00 → 2025-05-17 23:45:00


In [9]:
from sklearn.preprocessing import StandardScaler

## 6. Feature Scaling

StandardScaler is used as the initial scaling method.

The scaler is fitted only on the training data to prevent information leakage from validation and test observations.

The same fitted scaler is then applied to validation and test data.

In [10]:
feature_columns = [
    col for col in numeric_columns
    if col != "Potability"
]

scaler = StandardScaler()

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_columns] = scaler.fit_transform(
    train_df[feature_columns]
)

val_scaled[feature_columns] = scaler.transform(
    val_df[feature_columns]
)

test_scaled[feature_columns] = scaler.transform(
    test_df[feature_columns]
)

print("Scaling completed.")

Scaling completed.


In [11]:
output_dir = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed"

train_scaled.to_csv(
    output_dir + r"\train_v1.csv",
    index=False
)

val_scaled.to_csv(
    output_dir + r"\validation_v1.csv",
    index=False
)

test_scaled.to_csv(
    output_dir + r"\test_v1.csv",
    index=False
)

print("Processed Dataset V1 saved successfully.")

Processed Dataset V1 saved successfully.


In [12]:
print("TRAIN")
print(train_scaled.shape)
print(train_scaled["timestamp"].min())
print(train_scaled["timestamp"].max())

print("\nVALIDATION")
print(val_scaled.shape)
print(val_scaled["timestamp"].min())
print(val_scaled["timestamp"].max())

print("\nTEST")
print(test_scaled.shape)
print(test_scaled["timestamp"].min())
print(test_scaled["timestamp"].max())

TRAIN
(2291, 11)
2025-01-01 01:00:00
2025-04-07 00:45:00

VALIDATION
(491, 11)
2025-04-07 01:45:00
2025-04-27 14:45:00

TEST
(491, 11)
2025-04-27 15:45:00
2025-05-17 23:45:00


In [13]:
print("\nChronological checks:")

print(
    "Train ends before validation:",
    train_scaled["timestamp"].max()
    <= val_scaled["timestamp"].min()
)

print(
    "Validation ends before test:",
    val_scaled["timestamp"].max()
    <= test_scaled["timestamp"].min()
)

print(
    "Train feature means approximately 0:"
)

display(
    train_scaled[feature_columns]
    .mean()
    .round(3)
)


Chronological checks:
Train ends before validation: True
Validation ends before test: True
Train feature means approximately 0:


ph                -0.0
Hardness           0.0
Solids            -0.0
Chloramines       -0.0
Sulfate           -0.0
Conductivity      -0.0
Organic_carbon     0.0
Trihalomethanes   -0.0
Turbidity         -0.0
dtype: float64

# Day 11 – Assumptions and Methodology

## Assumptions

1. The timestamps are synthetic because the supplied dataset does not contain real temporal information.
2. The original raw dataset is preserved and is not modified.
3. The timestamped dataset generated during Day 8 is treated as the input to the temporal preprocessing pipeline.
4. Duplicate timestamps are handled deterministically by retaining the first record.
5. Missing numeric values are initially handled using forward filling.
6. Invalid parameter values are removed using basic domain constraints.
7. Data is split chronologically into 70% training, 15% validation, and 15% testing.
8. Random train/test splitting is avoided to reduce temporal data leakage.
9. Feature scaling is fitted only on the training data.
10. The same fitted scaler is applied to validation and test data.
11. The preprocessing pipeline is designed to be reusable for future anomaly-detection and forecasting experiments.
12. The generated temporal patterns should not be interpreted as actual environmental or sensor behaviour.

## Day 11 Deliverable

The Day 11 preprocessing pipeline produces:

- Cleaned timestamped data
- Chronological train/validation/test datasets
- Scaled numerical features
- Reproducible preprocessing steps
- Preserved raw and processed datasets

These outputs will be used as inputs for subsequent anomaly-detection and forecasting experiments.